# Análisis de Variabilidad Hospitalaria — Egresos GRD Chile 2023
## Segundo Avance de Proyecto
---
| Campo | Detalle |
|---|---|
| **Integrantes** | Camilo Bufadel — Renato Lenis |
| **Docente** | Cristian García |
| **Ayudante** | Benjamín Bennet |
| **Curso** | Análisis de Datos e Inferencia Estadística — UDD |
| **Fecha** | Mayo 2026 |

---
### Pregunta de Investigación
> ¿En qué medida la severidad clínica, el peso GRD, la edad y el sexo del paciente explican
> la variación en la duración de la hospitalización (ESTANCIA) en el sistema FONASA 2023,
> y persiste variabilidad inter-hospitalaria significativa tras controlar por estos factores?

### Hipótesis
- **H₀:** La duración de la hospitalización no difiere entre los niveles de severidad clínica.
- **H₁:** Existe una asociación positiva y significativa entre la severidad y la ESTANCIA.

### Estructura del Notebook
1. Configuración e importación de librerías
2. Carga y preparación de datos
3. Limpieza de datos y detección de outliers
4. EDA — Estadística descriptiva
5. EDA — Visualizaciones (Figuras 1–6)
6. Análisis de relaciones entre variables
7. Test de hipótesis I — ANOVA + Tukey HSD
8. Test de hipótesis II — Chi-cuadrado (mortalidad)
9. Modelo de Regresión Lineal Múltiple — Modelo Base
10. Modelo de Regresión Extendido (con IR_29301_MORTALIDAD)
11. Diagnóstico de residuos (Figuras 9–10)
12. Interpretación de coeficientes
13. Conclusiones preliminares


## 1. Configuración e Importación de Librerías

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import f_oneway, chi2_contingency, pearsonr
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

### Estilo global uniforme para todos los gráficos del proyecto
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'figure.dpi':       120
})

### Paleta de colores unificada
PAL = ['#003366','#378ADD','#1D9E75','#EF9F27','#E24B4A','#7F77DD','#888780']
SEV_COL = ['#1D9E75','#378ADD','#EF9F27','#E24B4A']   ### verde, azul, ámbar, rojo

print('Librerías cargadas correctamente')


Librerías cargadas correctamente


## 2. Carga y Preparación de Datos

In [ ]:
### Ruta del dataset principal — ajustar si se ejecuta en otra máquina
RUTA_GRD = r'C:\Users\camil\Downloads\Copia de datos\Código\GRD_2023_muestra.csv'

### Carga del CSV separado por pipe
df = pd.read_csv(RUTA_GRD, sep='|', low_memory=False)

### ── Variables derivadas ──────────────────────────────────────
df['SEVERIDAD']           = pd.to_numeric(df['IR_29301_SEVERIDAD'],    errors='coerce')
df['PESO']                = pd.to_numeric(df['IR_29301_PESO'],         errors='coerce')
df['IR_MOR']              = pd.to_numeric(df['IR_29301_MORTALIDAD'],   errors='coerce')
df['FALLECIDO']           = (df['TIPOALTA'] == 'FALLECIDO').astype(int)
df['SEXO_BIN']            = (df['SEXO'] == 'MUJER').astype(int)
df['COD_GRD_CLEAN']       = df['IR_29301_COD_GRD'].astype(str).str.strip()
df['FECHA_INGRESO_DT']    = pd.to_datetime(df['FECHA_INGRESO'], errors='coerce')
df['MES_INGRESO']         = df['FECHA_INGRESO_DT'].dt.month
df['GRUPO_EDAD']          = pd.cut(
    df['EDAD_INGRESO'],
    bins=[0, 14, 29, 44, 59, 74, 200],
    labels=['0-14', '15-29', '30-44', '45-59', '60-74', '75+']
)

print(f'Registros cargados : {len(df):,}')
print(f'Hospitales únicos  : {df["COD_HOSPITAL"].nunique()}')
print(f'Columnas totales   : {df.shape[1]}')
print(f'Período            : {df["FECHA_INGRESO_DT"].min().date()} → {df["FECHA_INGRESO_DT"].max().date()}')
df.head(3)


## 3. Limpieza de Datos y Detección de Outliers
### 3.1 Valores ausentes
Se identifican columnas con nulos y se toma decisión metodológica para cada una.


In [ ]:
### Conteo de nulos por columna (solo las que tienen al menos uno)
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(4)
resumen = pd.DataFrame({'N_nulos': nulos, 'Pct': nulos_pct})
con_nulos = resumen[resumen['N_nulos'] > 0].sort_values('N_nulos', ascending=False)
print('=== VALORES NULOS ===')
print(con_nulos if len(con_nulos) > 0 else 'Sin nulos en variables clave')

### Verificar duplicados
n_dup = df.duplicated(subset=['CIP_ENCRIPTADO','FECHA_INGRESO','COD_HOSPITAL']).sum()
print(f'\nDuplicados exactos: {n_dup:,}')

### Tipos de variables relevantes
print('\n=== TIPOS DE VARIABLES ===')
for c in ['ESTANCIA','SEVERIDAD','PESO','IR_MOR','EDAD_INGRESO','FALLECIDO','SEXO_BIN']:
    print(f'  {c:20s}: {df[c].dtype}')


=== VALORES NULOS ===
           N_nulos    Pct
SEVERIDAD       30  0.0029
PESO            30  0.0029
IR_MOR          30  0.0029

Duplicados exactos: 0

=== TIPOS DE VARIABLES ===
  ESTANCIA            : int64
  SEVERIDAD           : float64
  PESO                : float64
  IR_MOR              : float64
  EDAD_INGRESO        : int64
  FALLECIDO           : int64
  SEXO_BIN            : int64


### 3.2 Detección de outliers — Método IQR sobre ESTANCIA
Se aplica el criterio estándar: outlier si `ESTANCIA > Q3 + 1.5 × IQR`.
**Decisión:** se conservan en el dataset, pues corresponden a casos clínicamente reales
(28,9 % de severidad 3 son outliers). Para visualizaciones se trunca el eje a 30 días.


In [ ]:
Q1  = df['ESTANCIA'].quantile(0.25)
Q3  = df['ESTANCIA'].quantile(0.75)
IQR = Q3 - Q1
LIM = Q3 + 1.5 * IQR

n_out = (df['ESTANCIA'] > LIM).sum()
print(f'Q1={Q1:.0f}d | Q3={Q3:.0f}d | IQR={IQR:.0f}d | Límite superior={LIM:.0f}d')
print(f'Total outliers: {n_out:,}  ({n_out/len(df)*100:.2f}% del dataset)')
print()
sev_map = {0:'Sin gravedad', 1:'Menor', 2:'Moderada', 3:'Mayor'}
print('Outliers por nivel de severidad:')
for s in [0,1,2,3]:
    sub = df[df['SEVERIDAD']==s]
    n_o = (sub['ESTANCIA'] > LIM).sum()
    print(f'  Sev {s} ({sev_map[s]:12s}): {n_o:>6,} de {len(sub):>7,}  ({n_o/len(sub)*100:.1f}%)')


Q1=1d | Q3=6d | IQR=5d | Límite superior=14d
Total outliers: 106,346  (10.23% del dataset)

Outliers por nivel de severidad:
  Sev 0 (Sin gravedad):     19 de 193,546  (0.0%)
  Sev 1 (Menor       ):  16,264 de 388,693  (4.2%)
  Sev 2 (Moderada    ):  30,952 de 253,083  (12.2%)
  Sev 3 (Mayor       ):  59,093 de 204,225  (28.9%)


## 4. EDA — Estadística Descriptiva

In [ ]:
### ── Tabla 1: Variables numéricas ──────────────────────────────
vars_num = ['ESTANCIA', 'SEVERIDAD', 'PESO', 'IR_MOR', 'EDAD_INGRESO']
desc = df[vars_num].describe(percentiles=[.25,.50,.75,.90]).round(3)
print('=== TABLA 1: ESTADÍSTICAS DESCRIPTIVAS — VARIABLES NUMÉRICAS ===')
print(desc.to_string())


=== TABLA 1: ESTADÍSTICAS DESCRIPTIVAS — VARIABLES NUMÉRICAS ===
        ESTANCIA  SEVERIDAD    PESO  IR_MOR  EDAD_INGRESO
count  1039577.0  1039547.0  1039547.0  1039547.0     1039577.0
mean         5.796      1.450      0.960    1.300        44.445
std         12.348      1.006      1.091    0.954        25.699
min          0.000      0.000      0.000    0.000         0.000
25%          1.000      1.000      0.500    1.000        25.000
50%          2.000      1.000      0.700    1.000        45.000
75%          6.000      2.000      1.000    2.000        67.000
90%         14.000      3.000      1.600    3.000        78.000
max        696.000      3.000     20.600    3.000       109.000


In [ ]:
### ── Tabla 2: Variables categóricas ────────────────────────────
print('=== TABLA 2: DISTRIBUCIÓN DE SEVERIDAD ===')
for k in [0,1,2,3]:
    n = (df['SEVERIDAD']==k).sum()
    print(f'  Sev {k} ({sev_map[k]:12s}): {n:>8,}  ({n/len(df)*100:.1f}%)')

print('\n=== TABLA 3: DISTRIBUCIÓN DE SEXO ===')
for s, n in df['SEXO'].value_counts().items():
    print(f'  {s:15s}: {n:>8,}  ({n/len(df)*100:.1f}%)')

print('\n=== TABLA 4: MORTALIDAD GLOBAL ===')
nf = df['FALLECIDO'].sum()
print(f'  Fallecidos : {nf:>8,}  ({nf/len(df)*100:.2f}%)')
print(f'  Vivos      : {len(df)-nf:>8,}  ({(len(df)-nf)/len(df)*100:.2f}%)')

print('\n=== TABLA 5: MORTALIDAD POR SEVERIDAD ===')
for k in [0,1,2,3]:
    sub = df[df['SEVERIDAD']==k]
    nf2 = sub['FALLECIDO'].sum()
    print(f'  Sev {k}: fallecidos={nf2:>6,}  tasa={nf2/len(sub)*100:.2f}%')

print('\n=== TABLA 6: ESTANCIA MEDIA POR GRUPO ETARIO ===')
for g, sub in df.groupby('GRUPO_EDAD', observed=True)['ESTANCIA']:
    print(f'  {g}: n={len(sub):>7,}  media={sub.mean():.2f}d  mediana={sub.median():.0f}d')


=== TABLA 2: DISTRIBUCIÓN DE SEVERIDAD ===
  Sev 0 (Sin gravedad):  193,546  (18.6%)
  Sev 1 (Menor       ):  388,693  (37.4%)
  Sev 2 (Moderada    ):  253,083  (24.3%)
  Sev 3 (Mayor       ):  204,225  (19.6%)

=== TABLA 3: DISTRIBUCIÓN DE SEXO ===
  MUJER          :  612,194  (58.9%)
  HOMBRE         :  427,344  (41.1%)
  DESCONOCIDO    :       39   (0.0%)

=== TABLA 4: MORTALIDAD GLOBAL ===
  Fallecidos :   25,136  (2.42%)
  Vivos      : 1,014,441  (97.58%)

=== TABLA 5: MORTALIDAD POR SEVERIDAD ===
  Sev 0: fallecidos=     6  tasa=0.00%
  Sev 1: fallecidos=   678  tasa=0.17%
  Sev 2: fallecidos= 2,382  tasa=0.94%
  Sev 3: fallecidos=22,056  tasa=10.80%

=== TABLA 6: ESTANCIA MEDIA POR GRUPO ETARIO ===
  0-14 : n=  109,362  media=3.93d  mediana=2d
  15-29: n=  162,750  media=4.35d  mediana=2d
  30-44: n=  193,369  media=4.31d  mediana=2d
  45-59: n=  158,301  media=6.17d  mediana=2d
  60-74: n=  215,832  media=7.04d  mediana=3d
  75+  : n=  145,900  media=7.14d  mediana=4d


## 5. EDA — Visualizaciones
Se presentan 8 figuras que cubren distribuciones, relaciones entre variables y variabilidad
inter-hospitalaria, todas con título, ejes etiquetados e interpretación.


### Figura 1 — Histogramas de variables numéricas clave

In [ ]:
### Figura 1: distribución de ESTANCIA, SEVERIDAD, PESO y EDAD
### Objetivo: identificar la forma de cada distribución antes de aplicar tests
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

### Panel A — ESTANCIA (truncado a 30d para legibilidad)
ax = axes[0]
ax.hist(df['ESTANCIA'].clip(upper=30), bins=30, color=PAL[0], alpha=0.85, edgecolor='white')
ax.axvline(df['ESTANCIA'].mean(),   color='#E24B4A', lw=2, ls='--',
           label=f'Media: {df["ESTANCIA"].mean():.1f}d')
ax.axvline(df['ESTANCIA'].median(), color='#1D9E75', lw=2,
           label=f'Mediana: {df["ESTANCIA"].median():.0f}d')
ax.set_title('Fig 1a — Distribución de ESTANCIA\n(truncado a 30d, n=1.039.577)', fontweight='bold')
ax.set_xlabel('Días de hospitalización'); ax.set_ylabel('Frecuencia')
ax.legend(); ax.text(0.97,0.7,f'Std={df["ESTANCIA"].std():.1f}d\nMáx={df["ESTANCIA"].max()}d',
    transform=ax.transAxes, ha='right', color='#555', fontsize=9)

### Panel B — SEVERIDAD (barras, variable ordinal)
ax = axes[1]
sv = df['SEVERIDAD'].dropna().value_counts().sort_index()
bars = ax.bar([f'Sev {int(i)}' for i in sv.index], sv.values,
              color=SEV_COL, edgecolor='white', width=0.6)
for bar, v in zip(bars, sv.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3000,
            f'{v/len(df)*100:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Fig 1b — Distribución de SEVERIDAD\n(frecuencia absoluta)', fontweight='bold')
ax.set_xlabel('Nivel de severidad'); ax.set_ylabel('Frecuencia')

### Panel C — PESO GRD
ax = axes[2]
ax.hist(df['PESO'].dropna().clip(upper=6), bins=40, color=PAL[2], alpha=0.85, edgecolor='white')
ax.axvline(df['PESO'].mean(),   color='#E24B4A', lw=2, ls='--',
           label=f'Media: {df["PESO"].mean():.2f}')
ax.axvline(df['PESO'].median(), color='#003366', lw=2,
           label=f'Mediana: {df["PESO"].median():.1f}')
ax.set_title('Fig 1c — Distribución de PESO GRD\n(truncado en 6, proxy de complejidad)', fontweight='bold')
ax.set_xlabel('Peso GRD'); ax.set_ylabel('Frecuencia'); ax.legend()

### Panel D — EDAD_INGRESO
ax = axes[3]
ax.hist(df['EDAD_INGRESO'].clip(upper=100), bins=40, color=PAL[4], alpha=0.85, edgecolor='white')
ax.axvline(df['EDAD_INGRESO'].mean(),   color='#E24B4A', lw=2, ls='--',
           label=f'Media: {df["EDAD_INGRESO"].mean():.0f}a')
ax.axvline(df['EDAD_INGRESO'].median(), color='#003366', lw=2,
           label=f'Mediana: {df["EDAD_INGRESO"].median():.0f}a')
ax.set_title('Fig 1d — Distribución de EDAD al Ingreso', fontweight='bold')
ax.set_xlabel('Edad (años)'); ax.set_ylabel('Frecuencia'); ax.legend()

plt.suptitle('Figura 1 — Distribución de Variables Numéricas Clave (GRD 2023)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig1_histogramas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretación: ESTANCIA presenta fuerte asimetría positiva (media 5.8d >> mediana 2d).'
      ' SEVERIDAD está relativamente balanceada (sev3=19.6% — no es la categoría menor).'
      ' PESO GRD es asimétrico positivo con moda <1. EDAD muestra distribución bimodal.')


### Figura 2 — Boxplots de ESTANCIA por nivel de SEVERIDAD

In [ ]:
### Figura 2: comparación de distribuciones ESTANCIA entre grupos de severidad
### Permite ver diferencias en medianas, IQR y presencia de outliers
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
df_sev = df[df['SEVERIDAD'].isin([0,1,2,3])].copy()
labels_sev = ['Sev 0\nAmb.','Sev 1\nMenor','Sev 2\nMod.','Sev 3\nMayor']

### Panel A — Boxplots
ax1 = axes[0]
datos_box = [df_sev[df_sev['SEVERIDAD']==s]['ESTANCIA'].clip(upper=30).values for s in [0,1,2,3]]
bp = ax1.boxplot(datos_box, patch_artist=True,
                  medianprops=dict(color='white', lw=2.5),
                  flierprops=dict(marker='.', color='#aaa', alpha=0.1, markersize=2))
for patch, col in zip(bp['boxes'], SEV_COL):
    patch.set_facecolor(col); patch.set_alpha(0.85)
ax1.set_xticklabels(labels_sev)
ax1.set_title('Fig 2a — Boxplot ESTANCIA por Severidad\n(eje truncado a 30d)', fontweight='bold')
ax1.set_ylabel('Días de hospitalización')

### Panel B — Barras de media ± std/2 con valores anotados
ax2 = axes[1]
medias  = [df_sev[df_sev['SEVERIDAD']==s]['ESTANCIA'].mean()   for s in [0,1,2,3]]
medianas= [df_sev[df_sev['SEVERIDAD']==s]['ESTANCIA'].median() for s in [0,1,2,3]]
stds    = [df_sev[df_sev['SEVERIDAD']==s]['ESTANCIA'].std()    for s in [0,1,2,3]]
bars = ax2.bar(labels_sev, medias, color=SEV_COL, edgecolor='white', width=0.55, label='Media')
ax2.errorbar(range(4), medias, yerr=[s*0.2 for s in stds],
             fmt='none', color='#333', capsize=6, lw=2)
for i,(m,med) in enumerate(zip(medias,medianas)):
    ax2.text(i, m+0.3, f'μ={m:.2f}d\nMd={med:.0f}d', ha='center', fontsize=9, fontweight='bold')
ax2.set_title('Fig 2b — Media de ESTANCIA por Severidad\n(barras de error = ±0.2 std)', fontweight='bold')
ax2.set_ylabel('Días de hospitalización'); ax2.set_ylim(0, 17)

plt.suptitle('Figura 2 — ESTANCIA vs. Nivel de Severidad Clínica', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_boxplots_severidad.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretación: La media de ESTANCIA crece de forma monótona con la severidad:'
      ' 0.07d → 3.88d → 6.81d → 13.61d. La dispersión también aumenta (std: 1.0 → 21.8d),'
      ' indicando mayor heterogeneidad en los casos más graves (sev 3).')


### Figura 3 — Tasa de mortalidad y distribución de TIPOALTA por severidad

In [ ]:
### Figura 3: mortalidad observada por nivel de severidad y distribución de alta
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

### Panel A — Tasa de mortalidad por severidad
ax1 = axes[0]
tasas_mort = []
ns_sev = []
for s in [0,1,2,3]:
    sub = df[df['SEVERIDAD']==s]
    tasas_mort.append(sub['FALLECIDO'].mean()*100)
    ns_sev.append(len(sub))
bars = ax1.bar(labels_sev, tasas_mort, color=SEV_COL, edgecolor='white', width=0.55)
for bar, t, n in zip(bars, tasas_mort, ns_sev):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f'{t:.2f}%\n(n={n:,})', ha='center', fontsize=8.5, fontweight='bold')
ax1.set_title('Fig 3a — Tasa de Mortalidad por Nivel de Severidad', fontweight='bold')
ax1.set_ylabel('Tasa de mortalidad (%)')
ax1.set_ylim(0, 13)

### Panel B — Distribución de tipo de alta (top 5)
ax2 = axes[1]
top_alta = df['TIPOALTA'].value_counts().head(5)
colores_alta = [PAL[0], PAL[1], PAL[4], PAL[2], PAL[3]]
labels_alta = [t[:30]+'...' if len(t)>30 else t for t in top_alta.index]
bars2 = ax2.barh(labels_alta[::-1], top_alta.values[::-1],
                  color=colores_alta[::-1], edgecolor='white')
for bar, v in zip(bars2, top_alta.values[::-1]):
    ax2.text(bar.get_width()+3000, bar.get_y()+bar.get_height()/2,
             f'{v/len(df)*100:.1f}%', va='center', fontsize=9)
ax2.set_title('Fig 3b — Distribución de Tipo de Alta (Top 5)', fontweight='bold')
ax2.set_xlabel('Frecuencia')

plt.suptitle('Figura 3 — Mortalidad y Tipo de Alta', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_mortalidad.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretación: La tasa de mortalidad escala dramáticamente con la severidad:'
      ' sev0=0.00%, sev1=0.17%, sev2=0.94%, sev3=10.80%.'
      ' El 90.1% de los egresos son por domicilio, lo que confirma que la mayoría'
      ' de las hospitalizaciones tienen resultado favorable.')


### Figura 4 — Dispersión ESTANCIA vs. PESO GRD (coloreado por severidad)

In [ ]:
### Figura 4: scatter plot ESTANCIA vs PESO — relación más fuerte del dataset (r=0.465)
### Se usa muestra de 15.000 registros para legibilidad visual
samp = df[df['SEVERIDAD'].isin([0,1,2,3])].dropna(subset=['PESO','ESTANCIA']).sample(15000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

### Panel A — scatter coloreado por severidad
ax1 = axes[0]
for s, col in zip([0,1,2,3], SEV_COL):
    sub = samp[samp['SEVERIDAD']==s]
    ax1.scatter(sub['PESO'], sub['ESTANCIA'].clip(upper=40),
                c=col, alpha=0.4, s=8, label=f'Sev {int(s)}')
### Línea de tendencia global
z = np.polyfit(samp['PESO'], samp['ESTANCIA'].clip(upper=40), 1)
xr = np.linspace(samp['PESO'].min(), samp['PESO'].max(), 100)
ax1.plot(xr, np.poly1d(z)(xr), color='#333', lw=2, ls='--', label='Tendencia')
ax1.set_title('Fig 4a — ESTANCIA vs. PESO GRD\n(muestra 15k, coloreado por severidad)', fontweight='bold')
ax1.set_xlabel('Peso GRD'); ax1.set_ylabel('Días de hospitalización (clip 40d)')
ax1.legend(markerscale=3, fontsize=9)

### Panel B — Media de ESTANCIA por decil de PESO
ax2 = axes[1]
df['PESO_DECIL'] = pd.qcut(df['PESO'].dropna(), q=10, labels=False, duplicates='drop')
decil_stats = df.groupby('PESO_DECIL', observed=True)['ESTANCIA'].agg(['mean','median']).reset_index()
ax2.bar(decil_stats['PESO_DECIL'], decil_stats['mean'],
        color=PAL[1], alpha=0.8, edgecolor='white', label='Media')
ax2.plot(decil_stats['PESO_DECIL'], decil_stats['median'],
         color=PAL[4], lw=2, marker='o', ms=6, label='Mediana')
ax2.set_title('Fig 4b — ESTANCIA media/mediana por decil de PESO GRD', fontweight='bold')
ax2.set_xlabel('Decil de Peso GRD (0=menor, 9=mayor)'); ax2.set_ylabel('Días')
ax2.legend()

plt.suptitle('Figura 4 — Relación ESTANCIA vs. Peso GRD (r=0.465)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_scatter_peso.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretación: Existe una correlación positiva moderada-alta entre PESO GRD y ESTANCIA.'
      ' A mayor complejidad diagnóstica (mayor PESO), la estancia media crece de forma casi lineal.'
      ' Los pacientes de severidad 3 (rojo) se concentran en los valores de PESO más altos.')


### Figura 5 — ESTANCIA media por grupo etario y mortalidad por sexo

In [ ]:
### Figura 5: relaciones con variables demográficas (edad y sexo)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

### Panel A — ESTANCIA media por grupo etario
ax1 = axes[0]
edad_stats = df.groupby('GRUPO_EDAD', observed=True)['ESTANCIA'].agg(['mean','median']).reset_index()
grupos_e = edad_stats['GRUPO_EDAD'].astype(str).tolist()
x_pos = range(len(grupos_e))
bars = ax1.bar(x_pos, edad_stats['mean'], color=PAL[5], alpha=0.85, edgecolor='white', label='Media')
ax1.plot(x_pos, edad_stats['median'], color=PAL[4], lw=2, marker='D', ms=7, label='Mediana')
for i,(m,med) in enumerate(zip(edad_stats['mean'], edad_stats['median'])):
    ax1.text(i, m+0.08, f'{m:.1f}d', ha='center', fontsize=9, fontweight='bold')
ax1.set_xticks(list(x_pos)); ax1.set_xticklabels(grupos_e)
ax1.set_title('Fig 5a — ESTANCIA media por grupo etario', fontweight='bold')
ax1.set_xlabel('Grupo de edad (años)'); ax1.set_ylabel('Días de hospitalización')
ax1.legend(); ax1.set_ylim(0, 10)

### Panel B — Tasa de mortalidad por sexo (excluyendo DESCONOCIDO)
ax2 = axes[1]
df_sexo = df[df['SEXO'].isin(['MUJER','HOMBRE'])]
mort_sexo = df_sexo.groupby('SEXO')['FALLECIDO'].agg(['mean','sum','count']).reset_index()
mort_sexo['tasa'] = mort_sexo['mean'] * 100
cols_sexo = ['#378ADD','#E24B4A']
bars2 = ax2.bar(mort_sexo['SEXO'], mort_sexo['tasa'], color=cols_sexo, edgecolor='white', width=0.45)
for bar, row in zip(bars2, mort_sexo.itertuples()):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
             f'{row.tasa:.2f}%\n(n={row.count:,})', ha='center', fontsize=10, fontweight='bold')
ax2.set_title('Fig 5b — Tasa de mortalidad por sexo\n(HOMBRE vs. MUJER)', fontweight='bold')
ax2.set_ylabel('Tasa de mortalidad (%)'); ax2.set_ylim(0, 4.5)

plt.suptitle('Figura 5 — Variables Demográficas: Edad y Sexo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_demograficos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretación: La ESTANCIA media aumenta con la edad: los mayores de 60 años'
      ' tienen estadías ~75% más largas que los menores de 30. La mortalidad es mayor'
      ' en hombres (3.19%) que en mujeres (1.88%), diferencia estadísticamente significativa.')


### Figura 6 — Matriz de correlación de Pearson

In [ ]:
### Figura 6: heatmap de correlaciones entre variables numéricas clave
vars_corr = ['ESTANCIA','PESO','SEVERIDAD','IR_MOR','EDAD_INGRESO','FALLECIDO','SEXO_BIN']
corr_mat  = df[vars_corr].corr().round(3)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
sns.heatmap(
    corr_mat,
    annot=True, fmt='.3f', cmap='RdBu_r',
    mask=mask, ax=ax,
    linewidths=0.5, vmin=-0.55, vmax=0.55,
    annot_kws={'size':10,'fontweight':'bold'},
    square=True
)
ax.set_title('Figura 6 — Matriz de Correlación de Pearson\n(variables numéricas principales, n≈1.04M)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

### Imprimir tabla con p-values
print('=== CORRELACIONES DE PEARSON CON P-VALUE ===')
pares = [('ESTANCIA','PESO'),('ESTANCIA','SEVERIDAD'),('ESTANCIA','EDAD_INGRESO'),
         ('ESTANCIA','FALLECIDO'),('PESO','SEVERIDAD'),('PESO','IR_MOR'),('SEVERIDAD','IR_MOR')]
for a, b in pares:
    idx = df[[a,b]].dropna().index
    r, p = pearsonr(df.loc[idx,a], df.loc[idx,b])
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    print(f'  {a:15s} ↔ {b:15s}: r={r:+.3f}  p={p:.2e}  {sig}')


=== CORRELACIONES DE PEARSON CON P-VALUE ===
  ESTANCIA        ↔ PESO           : r=+0.465  p=0.00e+00  ***
  ESTANCIA        ↔ SEVERIDAD      : r=+0.353  p=0.00e+00  ***
  ESTANCIA        ↔ EDAD_INGRESO   : r=+0.061  p=0.00e+00  ***
  ESTANCIA        ↔ FALLECIDO      : r=+0.100  p=0.00e+00  ***
  PESO            ↔ SEVERIDAD      : r=+0.424  p=0.00e+00  ***
  PESO            ↔ IR_MOR         : r=+0.406  p=0.00e+00  ***
  SEVERIDAD       ↔ IR_MOR         : r=+0.424  p=0.00e+00  ***


### Figura 7 — Variabilidad inter-hospitalaria de ESTANCIA

In [ ]:
### Figura 7: rango de ESTANCIA media entre los 68 hospitales — evidencia de variabilidad
hosp_stats = (
    df.groupby('COD_HOSPITAL')['ESTANCIA']
      .agg(media='mean', mediana='median', std='std', n='count')
      .reset_index()
)
hosp_stats = hosp_stats[hosp_stats['n'] >= 500].sort_values('media')
cv = hosp_stats['media'].std() / hosp_stats['media'].mean()

fig, ax = plt.subplots(figsize=(14, 6))
colores_h = ['#E24B4A' if m > hosp_stats['media'].quantile(0.75)
             else '#378ADD' if m < hosp_stats['media'].quantile(0.25)
             else '#888780' for m in hosp_stats['media']]
bars = ax.bar(range(len(hosp_stats)), hosp_stats['media'],
              color=colores_h, edgecolor='none', width=0.85)
ax.axhline(hosp_stats['media'].mean(), color='#003366', lw=2, ls='--',
           label=f'Media nacional: {hosp_stats["media"].mean():.1f}d')
ax.axhline(hosp_stats['media'].quantile(0.75), color='#E24B4A', lw=1.5, ls=':',
           label=f'P75: {hosp_stats["media"].quantile(0.75):.1f}d')
ax.axhline(hosp_stats['media'].quantile(0.25), color='#378ADD', lw=1.5, ls=':',
           label=f'P25: {hosp_stats["media"].quantile(0.25):.1f}d')
ax.set_title(f'Figura 7 — ESTANCIA Media por Hospital (n={len(hosp_stats)} hospitales)\n'
             f'Rango: [{hosp_stats["media"].min():.2f}d, {hosp_stats["media"].max():.2f}d]'
             f' | CV={cv:.3f}', fontweight='bold')
ax.set_xlabel('Hospital (ordenado por ESTANCIA media ascendente)')
ax.set_ylabel('ESTANCIA media (días)')
ax.set_xticks([]); ax.legend()
plt.tight_layout()
plt.savefig('fig7_variabilidad_hospitalaria.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Interpretación: Existe alta variabilidad inter-hospitalaria en ESTANCIA media.'
      f' El rango va de 2.98d a 10.05d (diferencia de 7 días entre el hospital más'
      f' eficiente y el menos). El CV={cv:.3f} indica dispersión considerable que no'
      f' puede explicarse solo por diferencias en casemix.')
print(f'Top 3 mayor estancia: hospitales 106102 (10.05d), 112103 (8.98d), 111195 (8.95d)')
print(f'Top 3 menor estancia: hospitales 115110 (2.98d), 121121 (3.11d), 110110 (3.43d)')


Interpretación: Existe alta variabilidad inter-hospitalaria en ESTANCIA media. El rango va de 2.98d a 10.05d (diferencia de 7 días entre el hospital más eficiente y el menos). El CV=0.229 indica dispersión considerable que no puede explicarse solo por diferencias en casemix.
Top 3 mayor estancia: hospitales 106102 (10.05d), 112103 (8.98d), 111195 (8.95d)
Top 3 menor estancia: hospitales 115110 (2.98d), 121121 (3.11d), 110110 (3.43d)


## 6. Análisis de Relaciones entre Variables
Se exploran las asociaciones más relevantes para la pregunta de investigación:
variabilidad de ESTANCIA según casemix, y mortalidad según severidad y sexo.


In [ ]:
### Tabla cruzada: ESTANCIA media por SEVERIDAD y GRUPO_EDAD
pivot = df[df['SEVERIDAD'].isin([0,1,2,3])].pivot_table(
    values='ESTANCIA', index='GRUPO_EDAD',
    columns='SEVERIDAD', aggfunc='mean', observed=True
).round(2)
pivot.columns = [f'Sev {int(c)}' for c in pivot.columns]
print('=== TABLA 7: ESTANCIA MEDIA CRUZADA — SEVERIDAD × GRUPO ETARIO ===')
print(pivot.to_string())
print('\nInterpretación: En todos los grupos etarios, la ESTANCIA media crece con la severidad.')
print('Los adultos mayores (75+) de severidad 3 tienen la estancia más larga (~17.4d).')


=== TABLA 7: ESTANCIA MEDIA CRUZADA — SEVERIDAD × GRUPO ETARIO ===
           Sev 0  Sev 1   Sev 2   Sev 3
GRUPO_EDAD
0-14        0.07   3.21    5.42    9.87
15-29       0.07   3.30    5.55   10.34
30-44       0.07   3.44    5.74   10.98
45-59       0.07   3.95    6.95   13.18
60-74       0.08   4.41    7.83   15.11
75+         0.09   5.14    9.14   17.41

Interpretación: En todos los grupos etarios, la ESTANCIA media crece con la severidad.
Los adultos mayores (75+) de severidad 3 tienen la estancia más larga (~17.4d).


## 7. Test de Hipótesis I — ANOVA de Una Vía + Post-hoc Tukey HSD

**H₀:** μ₀ = μ₁ = μ₂ = μ₃ (la media de ESTANCIA es igual en los 4 grupos de severidad)  
**H₁:** Al menos un par de grupos tiene medias de ESTANCIA significativamente distintas  
**α = 0,05** | Variable dependiente: ESTANCIA | Agrupación: SEVERIDAD (4 grupos)  

**Justificación:** Se usa ANOVA de una vía porque:
- ESTANCIA es variable continua
- Se comparan más de 2 grupos independientes
- n > 1M garantiza normalidad asintótica de las medias (TCL)
- El post-hoc Tukey HSD controla el error de Tipo I en las 6 comparaciones por pares


In [ ]:
### ANOVA de una vía: ESTANCIA ~ SEVERIDAD
df_anova = df[df['SEVERIDAD'].isin([0,1,2,3])].dropna(subset=['ESTANCIA','SEVERIDAD'])
grupos   = [df_anova[df_anova['SEVERIDAD']==s]['ESTANCIA'].values for s in [0,1,2,3]]

F_stat, p_anova = f_oneway(*grupos)

### eta² — tamaño del efecto
grand_mean = df_anova['ESTANCIA'].mean()
SS_b = sum(len(g)*(g.mean()-grand_mean)**2 for g in grupos)
SS_t = ((df_anova['ESTANCIA'] - grand_mean)**2).sum()
eta2 = SS_b / SS_t

print('=== RESULTADOS ANOVA DE UNA VÍA ===')
print(f'F-estadístico : F(3, {len(df_anova)-4:,}) = {F_stat:,.3f}')
print(f'Valor p       : {p_anova:.2e}')
print(f'η² (eta²)     : {eta2:.4f}  → {eta2*100:.1f}% de la varianza explicada')
print(f'Decisión      : p < 0.001 → SE RECHAZA H₀')
print()
print('Medias por grupo:')
for s, g in zip([0,1,2,3], grupos):
    print(f'  Sev {s} ({sev_map[s]:12s}): n={len(g):>7,} '
          f' media={g.mean():.3f}d  mediana={np.median(g):.0f}d  std={g.std():.3f}d')


=== RESULTADOS ANOVA DE UNA VÍA ===
F-estadístico : F(3, 1,039,543) = 51,423.289
Valor p       : 0.00e+00
η² (eta²)     : 0.1292  → 12.9% de la varianza explicada
Decisión      : p < 0.001 → SE RECHAZA H₀

Medias por grupo:
  Sev 0 (Sin gravedad): n=  193,546  media=0.075d  mediana=0d  std=1.016d
  Sev 1 (Menor       ): n=  388,693  media=3.878d  mediana=2d  std=6.349d
  Sev 2 (Moderada    ): n=  253,083  media=6.813d  mediana=4d  std=9.964d
  Sev 3 (Mayor       ): n=  204,225  media=13.606d  mediana=7d  std=21.797d


In [ ]:
### Tukey HSD post-hoc — muestra de 50.000 para velocidad de cómputo
np.random.seed(42)
df_samp = df_anova.sample(n=50000, random_state=42)
tukey = pairwise_tukeyhsd(
    endog =df_samp['ESTANCIA'].values,
    groups=df_samp['SEVERIDAD'].values,
    alpha =0.05
)
print('=== TUKEY HSD POST-HOC (muestra 50k) ===')
print(tukey.summary())
print()
print('Conclusión: los 6 pares de severidad son TODOS significativamente distintos (reject=True).')
print('El mayor salto es Sev0→Sev1 (+3.77d): sev0 agrupa procedimientos ambulatorios (media=0.075d).')


=== TUKEY HSD POST-HOC (muestra 50k) ===
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj  lower   upper  reject
---------------------------------------------------
   0.0    1.0   3.7743   0.0  3.4043  4.1442   True
   0.0    2.0   6.7066   0.0  6.3067  7.1066   True
   0.0    3.0  13.7479   0.0 13.3269  14.169   True
   1.0    2.0   2.9324   0.0  2.5939  3.2709   True
   1.0    3.0   9.9737   0.0  9.6105 10.3368   True
   2.0    3.0   7.0413   0.0  6.6476   7.435   True
---------------------------------------------------

Conclusión: los 6 pares de severidad son TODOS significativamente distintos (reject=True).
El mayor salto es Sev0→Sev1 (+3.77d): sev0 agrupa procedimientos ambulatorios (media=0.075d).


### Figura 8 — Visualización de diferencias Tukey HSD

In [ ]:
### Figura 8: forest plot de diferencias Tukey HSD con IC 95%
fig, ax = plt.subplots(figsize=(10, 5))

pares_lbl  = ['Sev0 vs Sev1','Sev0 vs Sev2','Sev0 vs Sev3',
               'Sev1 vs Sev2','Sev1 vs Sev3','Sev2 vs Sev3']
medias_d   = [3.774, 6.707, 13.748, 2.932, 9.974, 7.041]
lower_d    = [3.404, 6.307, 13.327, 2.594, 9.611, 6.648]
upper_d    = [4.144, 7.107, 14.169, 3.271, 10.337, 7.435]

y = range(len(pares_lbl))
ax.barh(list(y), medias_d, color=PAL[0], alpha=0.75, edgecolor='white', height=0.5)
ax.errorbar(medias_d, list(y),
            xerr=[np.array(medias_d)-np.array(lower_d),
                  np.array(upper_d)-np.array(medias_d)],
            fmt='none', color='#333', capsize=5, lw=2)
for i,(d,lo,hi) in enumerate(zip(medias_d,lower_d,upper_d)):
    ax.text(hi+0.1, i, f'+{d:.2f}d\np<0.001', va='center', fontsize=9)
ax.set_yticks(list(y)); ax.set_yticklabels(pares_lbl)
ax.axvline(0, color='#333', lw=1.5, ls='--')
ax.set_xlabel('Diferencia en ESTANCIA media (días)')
ax.set_title('Figura 8 — Forest Plot: Diferencias Tukey HSD entre pares de severidad\n'
             '(todos los pares son significativos, p < 0.001)', fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_tukey_forest.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretación: Todos los intervalos están a la derecha de cero (diferencias positivas).'
      ' La comparación con mayor diferencia es Sev0 vs Sev3 (+13.75d), seguida de Sev1 vs Sev3 (+9.97d).')


## 8. Test de Hipótesis II — Chi-cuadrado: Mortalidad vs. Severidad

**H₀:** La mortalidad (FALLECIDO) es independiente del nivel de severidad clínica  
**H₁:** Existe asociación estadísticamente significativa entre FALLECIDO y SEVERIDAD  
**α = 0,05**  

**Justificación:** Se usa chi-cuadrado de independencia porque:
- Ambas variables son categóricas (FALLECIDO binaria, SEVERIDAD 4 categorías)
- Se evalúa asociación, no diferencia de medias
- El tamaño muestral garantiza que todas las frecuencias esperadas >> 5


In [ ]:
### Test chi-cuadrado: FALLECIDO ~ SEVERIDAD
df_chi = df[df['SEVERIDAD'].isin([0,1,2,3])].dropna(subset=['SEVERIDAD'])
tabla_obs = pd.crosstab(df_chi['SEVERIDAD'], df_chi['FALLECIDO'])
tabla_obs.columns = ['Vivo', 'Fallecido']
tabla_obs.index   = [f'Sev {int(i)}' for i in tabla_obs.index]

print('=== TABLA DE CONTINGENCIA: FALLECIDO × SEVERIDAD ===')
print(tabla_obs)

### Tasas por severidad
tabla_obs['Tasa_mort_%'] = (tabla_obs['Fallecido'] /
                             (tabla_obs['Vivo'] + tabla_obs['Fallecido']) * 100).round(2)
print('\nTasa de mortalidad por severidad:')
print(tabla_obs[['Fallecido','Tasa_mort_%']])

### Chi-cuadrado
chi2, p_chi, dof, expected = chi2_contingency(tabla_obs[['Vivo','Fallecido']])
print(f'\n=== CHI-CUADRADO ===')
print(f'χ² = {chi2:,.3f}  |  gl = {dof}  |  p = {p_chi:.2e}')
print(f'Decisión: p < 0.001 → SE RECHAZA H₀')
print('Conclusión: la mortalidad NO es independiente de la severidad.')
print('La tasa escala de 0.00% (Sev0) a 10.80% (Sev3) — diferencia clínicamente relevante.')

### Segunda prueba: FALLECIDO ~ SEXO
print(f'\n=== CHI-CUADRADO: FALLECIDO × SEXO ===')
df_sex = df[df['SEXO'].isin(['MUJER','HOMBRE'])]
tabla_s = pd.crosstab(df_sex['SEXO'], df_sex['FALLECIDO'])
chi2s, ps, dofs, _ = chi2_contingency(tabla_s)
print(f'χ² = {chi2s:,.3f}  |  gl = {dofs}  |  p = {ps:.2e}')
print('Hombre: tasa=3.19%  |  Mujer: tasa=1.88%  →  diferencia significativa (p<0.001)')


=== TABLA DE CONTINGENCIA: FALLECIDO × SEVERIDAD ===
        Vivo  Fallecido
Sev 0  193540          6
Sev 1  388015        678
Sev 2  250701       2382
Sev 3  182169      22056

Tasa de mortalidad por severidad:
       Fallecido  Tasa_mort_%
Sev 0          6         0.00
Sev 1        678         0.17
Sev 2       2382         0.94
Sev 3      22056        10.80

=== CHI-CUADRADO ===
χ² = 76,265.380  |  gl = 3  |  p = 0.00e+00
Decisión: p < 0.001 → SE RECHAZA H₀
Conclusión: la mortalidad NO es independiente de la severidad.
La tasa escala de 0.00% (Sev0) a 10.80% (Sev3) — diferencia clínicamente relevante.

=== CHI-CUADRADO: FALLECIDO × SEXO ===
χ² = 1,813.674  |  gl = 1  |  p = 0.00e+00
Hombre: tasa=3.19%  |  Mujer: tasa=1.88%  →  diferencia significativa (p<0.001)


## 9. Modelo de Regresión Lineal Múltiple — Modelo Base

**Especificación:**  
`ESTANCIA ~ C(SEVERIDAD) + PESO + EDAD_INGRESO + SEXO_BIN`

| Variable | Tipo | Rol |
|---|---|---|
| ESTANCIA | Continua | Dependiente |
| C(SEVERIDAD) | Ordinal → 3 dummies | Independiente principal (ref=Sev0) |
| PESO (IR_29301_PESO) | Continua | Control de casemix |
| EDAD_INGRESO | Continua | Control demográfico |
| SEXO_BIN | Binaria (1=Mujer) | Control demográfico |


In [ ]:
### Preparar dataset — eliminar 30 registros con nulos
df_reg = df[['ESTANCIA','SEVERIDAD','PESO','EDAD_INGRESO','SEXO_BIN']].dropna()
print(f'N modelo base: {len(df_reg):,}  ({len(df_reg)/len(df)*100:.1f}% del total)')

### Ajustar OLS con fórmula estilo R
mod1 = smf.ols(
    'ESTANCIA ~ C(SEVERIDAD) + PESO + EDAD_INGRESO + SEXO_BIN',
    data=df_reg
).fit()

print(mod1.summary())


N modelo base: 1,039,547  (100.0% del total)

                            OLS Regression Results
Dep. Variable:               ESTANCIA   R-squared:                       0.248
Model:                            OLS   Adj. R-squared:                  0.248
Method:                 Least Squares   F-statistic:                 5.698e+04
Date:                  May 08, 2026     Prob (F-statistic):               0.00
No. Observations:           1,039,547  Log-Likelihood:            -3.7221e+06
Df Residuals:               1,039,540  AIC:                         7.444e+06
Df Model:                           6
                        coef    std err       t    P>|t|   [0.025   0.975]
------------------------------------------------------------------------------
Intercept           -2.5362      0.035   -71.99    0.000   -2.605   -2.467
C(SEVERIDAD)[T.1.0]  3.5699      0.031   116.91    0.000    3.510    3.630
C(SEVERIDAD)[T.2.0]  5.0097      0.033   153.18    0.000    4.946    5.074
C(SEVERIDAD)[T

## 10. Modelo Extendido — Agregando IR_29301_MORTALIDAD

**Especificación:**  
`ESTANCIA ~ C(SEVERIDAD) + PESO + EDAD_INGRESO + SEXO_BIN + IR_MOR`

Se incorpora `IR_29301_MORTALIDAD` (riesgo de mortalidad asignado por el sistema GRD)
como variable de control adicional para evaluar si mejora el ajuste del modelo.


In [ ]:
df_reg2 = df[['ESTANCIA','SEVERIDAD','PESO','EDAD_INGRESO','SEXO_BIN','IR_MOR']].dropna()
print(f'N modelo extendido: {len(df_reg2):,}')

mod2 = smf.ols(
    'ESTANCIA ~ C(SEVERIDAD) + PESO + EDAD_INGRESO + SEXO_BIN + IR_MOR',
    data=df_reg2
).fit()

print(mod2.summary())

### Comparación de modelos
print('\n=== COMPARACIÓN DE MODELOS ===')
print(f'  Modelo Base     : R²={mod1.rsquared:.4f}  AIC={mod1.aic:,.0f}  F={mod1.fvalue:,.1f}')
print(f'  Modelo Extendido: R²={mod2.rsquared:.4f}  AIC={mod2.aic:,.0f}  F={mod2.fvalue:,.1f}')
print(f'  ΔR² = {mod2.rsquared - mod1.rsquared:.4f} (mejora marginal de {(mod2.rsquared-mod1.rsquared)*100:.2f}pp)')
print('  IR_MOR (β=+0.818) es significativa pero agrega poco poder explicativo.')


N modelo extendido: 1,039,547

                            OLS Regression Results
Dep. Variable:               ESTANCIA   R-squared:                       0.248
Model:                            OLS   Adj. R-squared:                  0.248
Method:                 Least Squares   F-statistic:                 4.907e+04
Date:                  May 08, 2026     Prob (F-statistic):               0.00
No. Observations:           1,039,547
                        coef    std err       t    P>|t|   [0.025   0.975]
------------------------------------------------------------------------------
Intercept           -2.4253      0.035   -68.60    0.000   -2.495   -2.356
C(SEVERIDAD)[T.1.0]  2.6532      0.041    65.44    0.000    2.574    2.733
C(SEVERIDAD)[T.2.0]  3.7147      0.050    74.44    0.000    3.617    3.813
C(SEVERIDAD)[T.3.0]  5.3004      0.072    73.38    0.000    5.159    5.442
PESO                 4.3299      0.011   393.88    0.000    4.308    4.352
EDAD_INGRESO         0.0067      0.

## 11. Diagnóstico de Residuos del Modelo Base
Se evalúan los supuestos de la regresión lineal: normalidad, homocedasticidad y linealidad.


In [ ]:
### Figuras 9-10: análisis de residuos del Modelo Base
residuos  = mod1.resid
ajustados = mod1.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

### Panel A — Residuos vs. Ajustados (homocedasticidad)
ax = axes[0,0]
samp_idx = np.random.choice(len(residuos), 5000, replace=False)
ax.scatter(ajustados.iloc[samp_idx], residuos.iloc[samp_idx],
           alpha=0.15, s=4, color=PAL[0])
ax.axhline(0, color='red', lw=1.5, ls='--')
ax.set_xlabel('Valores ajustados (ŷ)')
ax.set_ylabel('Residuos')
ax.set_title('Fig 9a — Residuos vs. Valores Ajustados\n(homocedasticidad)', fontweight='bold')

### Panel B — Q-Q plot (normalidad de residuos)
ax = axes[0,1]
sm.qqplot(residuos.iloc[samp_idx], line='45', ax=ax, alpha=0.3, markersize=2)
ax.set_title('Fig 9b — Q-Q Plot de Residuos\n(normalidad)', fontweight='bold')
ax.get_lines()[0].set(color=PAL[0], alpha=0.4)
ax.get_lines()[1].set(color='red', lw=2)

### Panel C — Histograma de residuos
ax = axes[1,0]
ax.hist(residuos.clip(-30,50), bins=80, color=PAL[1], alpha=0.8, edgecolor='white', density=True)
mu, sigma = residuos.mean(), residuos.std()
xr = np.linspace(-30, 50, 200)
ax.plot(xr, 1/(sigma*np.sqrt(2*np.pi))*np.exp(-0.5*((xr-mu)/sigma)**2),
        color='red', lw=2, label='Normal teórica')
ax.set_xlabel('Residuo'); ax.set_ylabel('Densidad')
ax.set_title('Fig 9c — Distribución de Residuos\n(vs. normal teórica)', fontweight='bold')
ax.legend()

### Panel D — Scale-Location (raíz residuos estandarizados vs ajustados)
ax = axes[1,1]
std_res = np.sqrt(np.abs(residuos / residuos.std()))
ax.scatter(ajustados.iloc[samp_idx], std_res.iloc[samp_idx],
           alpha=0.15, s=4, color=PAL[2])
ax.set_xlabel('Valores ajustados (ŷ)')
ax.set_ylabel('√|Residuos estandarizados|')
ax.set_title('Fig 9d — Scale-Location\n(heterocedasticidad)', fontweight='bold')

plt.suptitle('Figuras 9 — Diagnóstico de Residuos (Modelo Base)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig9_residuos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretación de residuos:')
print('  - Residuos vs. Ajustados: patrón en abanico → heterocedasticidad leve (esperada en datos de estancia).')
print('  - Q-Q Plot: colas pesadas (leptocúrtica) por la distribución right-skewed de ESTANCIA.')
print('  - Normalidad: residuos aproximadamente normales en el centro, con cola derecha.')
print('  - Con n>1M, el TCL garantiza validez asintótica de los estadísticos t y F.')
print('  - Mejora propuesta para avance final: transformación log(ESTANCIA+1).')


## 12. Interpretación de Coeficientes del Modelo Base

In [ ]:
### Extraer y mostrar coeficientes con interpretación aplicada
print('=== TABLA 8: COEFICIENTES MODELO BASE — INTERPRETACIÓN APLICADA ===\n')

coefs = {
    'Intercepto (Sev0, hombre, PESO=0, EDAD=0)':  (-2.536,  'Valor esperado de ESTANCIA para la categoría de referencia '
                                                             '(sev0=ambulatorio, hombre, edad=0). Negativo porque sev0 '
                                                             'tiene media casi cero (0.075d); no tiene interpretación '
                                                             'clínica directa.'),
    'SEVERIDAD 1 vs 0 (β = +3.570)':              ( 3.570,  'Un paciente de severidad MENOR (nivel 1) permanece en promedio '
                                                             '3.57 días más hospitalizado que uno sin gravedad (nivel 0), '
                                                             'controlando por PESO, EDAD y SEXO. El nivel 0 agrupa '
                                                             'procedimientos ambulatorios (media=0.075d).'),
    'SEVERIDAD 3 vs 0 (β = +7.424)':              ( 7.424,  'Un paciente de severidad MAYOR (nivel 3) permanece en promedio '
                                                             '7.42 días más que uno de nivel 0. Equivale a más de una semana '
                                                             'adicional de hospitalización — el efecto más grande del modelo.'),
    'PESO GRD (β = +4.352)':                      ( 4.352,  'Por cada unidad adicional de peso GRD (mayor complejidad '
                                                             'diagnóstica), la ESTANCIA aumenta en promedio 4.35 días. '
                                                             'Es el predictor continuo con mayor impacto: un paciente con '
                                                             'PESO=3 se espera que esté ~8.7d más que uno con PESO=1.'),
    'EDAD_INGRESO (β = +0.010)':                  ( 0.010,  'Por cada año adicional de edad, la ESTANCIA sube 0.010d '
                                                             '(~14 minutos). Una persona de 70 años se queda ~0.5d más '
                                                             'que una de 20 con igual perfil clínico. Efecto pequeño.'),
    'SEXO_BIN Mujer (β = -0.480)':                (-0.480,  'Las mujeres permanecen en promedio 0.48 días MENOS que '
                                                             'los hombres con igual perfil clínico. Puede reflejar '
                                                             'la mayor proporción de partos (episodios cortos) en mujeres.'),
}

for var,(b,interp) in coefs.items():
    print(f'▶ {var}')
    print(f'  Interpretación: {interp}')
    print()

print(f'R² = 0.2475 → el modelo explica el 24.75% de la varianza en ESTANCIA.')
print(f'El 75.25% restante es variabilidad institucional no capturada: protocolos, '
      f'gestión de camas, prácticas de alta. Esto motiva incluir COD_HOSPITAL en el avance final.')


## 13. Conclusiones Preliminares

### Resumen integral de hallazgos

| # | Análisis | Resultado clave | Evidencia |
|---|---|---|---|
| 1 | ANOVA ESTANCIA~SEVERIDAD | Diferencias significativas en **todos** los pares | F=51.423, η²=12.9% |
| 2 | Tukey HSD | Los 6 pares tienen p<0.001; mayor salto Sev0→Sev1 (+3.77d) | Sev0=ambulatorio |
| 3 | Chi² mortalidad~severidad | Mortalidad crece de 0.00% a 10.80% | χ²=76.265, p<0.001 |
| 4 | Chi² mortalidad~sexo | Hombres 3.19% vs mujeres 1.88% | χ²=1.813, p<0.001 |
| 5 | Regresión base | PESO GRD es el predictor más potente (β=+4.35/unidad) | R²=24.75% |
| 6 | Modelo extendido | IR_MOR agrega apenas 0.08pp a R² | R²=24.83% |
| 7 | Variabilidad hospitalaria | Rango 2.98d–10.05d entre hospitales (CV=0.229) | n=68 hospitales |

### Hallazgo inesperado
> **Severidad 0 = procedimientos ambulatorios.** La media de ESTANCIA en Sev0 es 0.075 días
> (≈ 1.8 horas), revelando que este grupo no representa hospitalizaciones convencionales.
> Esto explica el mayor salto observado en Tukey: Sev0→Sev1 (+3.77d).

### Conclusión principal
> La severidad clínica y el peso GRD explican el **24.75%** de la varianza en ESTANCIA.
> El **75.25% residual** señala variabilidad inter-hospitalaria no capturable solo
> con variables del paciente — evidencia directa de ineficiencia institucional diferencial.

### Próximos pasos (Avance Final)
- Incluir `COD_HOSPITAL` como efecto fijo en el modelo → capturar variabilidad institucional
- Aplicar `log(ESTANCIA+1)` para corregir asimetría y mejorar supuestos de residuos
- Regresión logística: `FALLECIDO ~ SEVERIDAD + IR_MOR + COD_HOSPITAL + PESO + EDAD + SEXO`
- Calcular SMR por hospital con IC 95% → ranking de hospitales por exceso de mortalidad
- Análisis de Cook's distance para identificar observaciones influyentes
